# objective loss contract v1

目标：把 `src/objective` 从 evaluation/promotion 扩展到稳定 loss contract，覆盖 MSE、classification、IC 和 ranking/Rank IC 的可最小化 loss 表达，并记录本次 `src/objective` 扩展原因。

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src").exists() and (candidate / "AGENTS.md").exists():
        ROOT = candidate
        break
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.core import build_architecture_extension_decision, write_architecture_extension_decision
from src.objective import compute_objective_losses

EXPERIMENT = "objective_loss_contract_v1"
CARD = Path("research") / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
CARD


PosixPath('research/objective_loss_contract_v1')

In [2]:
frame = pd.DataFrame(
    {
        "date": list(pd.date_range("2026-01-01", periods=3, freq="D")) * 2,
        "instrument": ["SOXL"] * 3 + ["SOXX"] * 3,
        "prediction": [0.03, -0.005, 0.01, 0.02, -0.02, -0.02],
        "target_return": [0.02, -0.01, 0.015, 0.01, -0.02, -0.01],
    }
)
losses = compute_objective_losses(frame, group_col="date")
(CARD / "objective_losses.json").write_text(json.dumps(losses, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
pd.DataFrame([losses]).drop(columns=["evaluation"]).to_csv(CARD / "objective_losses.csv", index=False, lineterminator="\n")
frame.to_csv(CARD / "objective_loss_frame.csv", index=False, lineterminator="\n")
losses


{'schema_version': 'objective.loss.v1',
 'metric_source_schema_version': 'objective.evaluation.v1',
 'optimization_direction': 'minimize',
 'reward_surface': 'retired',
 'objective_names': ['mse', 'classification', 'ic', 'ranking'],
 'row_count': 6,
 'mse_loss': 5.8333333333333326e-05,
 'classification_loss': 0.0,
 'ic_loss': 0.06606706099146509,
 'ranking_loss': 1.1102230246251565e-16,
 'rank_group_count': 3,
 'evaluation': {'schema_version': 'objective.evaluation.v1',
  'row_count': 6,
  'mse': 5.8333333333333326e-05,
  'classification_accuracy': 1.0,
  'ic': 0.9339329390085349,
  'rank_ic': 0.9999999999999999,
  'rank_group_count': 3}}

In [3]:
decision = build_architecture_extension_decision(
    decision_id="src-objective-loss-contract-v1",
    target_surface="src/objective",
    change_type="stable_loss_contract",
    reason=(
        "AGENTS.md 明确 src/objective 负责 loss、IC/ranking objective 和收益风险 utility；"
        "在 rank_ic_objective research evidence 完成后，需要 compute_objective_losses 作为稳定 loss contract，"
        "供模型训练、analysis 和 promotion 复用，同时继续保持 reward surface retired。"
    ),
    impact_scope=(
        "src/objective",
        "tests/test_objective_contract.py",
        "research/objective_loss_contract_v1",
        "research/rank_ic_objective_validation_v1",
    ),
    evidence_notebook="research/objective_loss_contract_v1/objective_loss_contract_v1.ipynb",
    verification_commands=(
        "PYTHONPATH=. /Users/rziiiii/dev/workspace/qlib_venv/bin/python -m pytest tests/test_objective_contract.py -q",
        "PYTHONPATH=. /Users/rziiiii/dev/workspace/qlib_venv/bin/python -m pytest tests -q",
    ),
    alternatives_considered=(
        "只保留 evaluate_prediction_objectives，不提供 loss contract",
        "把 Rank IC loss 只写在单次 notebook 中",
        "把 objective 方法写入 skill",
    ),
    constraints=(
        "不新增 RL reward surface",
        "research/production notebook 是证据面，src/objective 是稳定接口",
        "dashboard 仍只能读取 manifest/export sidecar",
    ),
    created_at="2026-06-19T22:00:00Z",
)
write_result = write_architecture_extension_decision(
    output_dir=CARD,
    decision=decision,
    registry_path=CARD / "architecture_extension_registry.json",
)
decision_payload = json.loads(Path(write_result["decision_path"]).read_text(encoding="utf-8"))
registry = json.loads(Path(write_result["registry_path"]).read_text(encoding="utf-8"))
review_text = Path(write_result["review_path"]).read_text(encoding="utf-8")
write_result


{'schema_version': 'core.architecture_extension_write_result.v1',
 'decision_path': 'research/objective_loss_contract_v1/src-objective-loss-contract-v1.json',
 'registry_path': 'research/objective_loss_contract_v1/architecture_extension_registry.json',
 'review_path': 'research/objective_loss_contract_v1/architecture_extension_review.md',
 'decision_hash': 'f0d8963f64a4028f390b353db4e71d4bfb3c13afd8a22c3f414dada870d223a3',
 'registry_hash': '32ad6bf04474feac03d4b2e700cee7d0f777c8d41205a7f691b79960306c7b71'}

In [4]:
summary = {
    "schema_version": "research.objective_loss_contract.v1",
    "status": "completed",
    "loss_schema_version": losses["schema_version"],
    "metric_source_schema_version": losses["metric_source_schema_version"],
    "optimization_direction": losses["optimization_direction"],
    "reward_surface": losses["reward_surface"],
    "objective_names": losses["objective_names"],
    "row_count": losses["row_count"],
    "mse_loss": losses["mse_loss"],
    "classification_loss": losses["classification_loss"],
    "ic_loss": losses["ic_loss"],
    "ranking_loss": losses["ranking_loss"],
    "rank_group_count": losses["rank_group_count"],
    "architecture_decision_id": decision_payload["decision_id"],
    "extension_requires_reason": decision_payload["extension_requires_reason"],
    "blocked_without_evidence": decision_payload["blocked_without_evidence"],
    "decision_hash": decision_payload["decision_hash"],
    "registry_hash": registry["registry_hash"],
    "artifacts": {
        "objective_losses": "objective_losses.json",
        "objective_losses_csv": "objective_losses.csv",
        "objective_loss_frame": "objective_loss_frame.csv",
        "architecture_decision": Path(write_result["decision_path"]).name,
        "architecture_registry": Path(write_result["registry_path"]).name,
        "architecture_review": Path(write_result["review_path"]).name,
    },
}

assert summary["loss_schema_version"] == "objective.loss.v1"
assert summary["reward_surface"] == "retired"
assert abs(summary["ranking_loss"]) < 1e-12
assert decision_payload["target_surface"] == "src/objective"
assert decision_payload["change_type"] == "stable_loss_contract"
assert "中文结论" in review_text

(CARD / "analysis_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
summary


{'schema_version': 'research.objective_loss_contract.v1',
 'status': 'completed',
 'loss_schema_version': 'objective.loss.v1',
 'metric_source_schema_version': 'objective.evaluation.v1',
 'optimization_direction': 'minimize',
 'reward_surface': 'retired',
 'objective_names': ['mse', 'classification', 'ic', 'ranking'],
 'row_count': 6,
 'mse_loss': 5.8333333333333326e-05,
 'classification_loss': 0.0,
 'ic_loss': 0.06606706099146509,
 'ranking_loss': 1.1102230246251565e-16,
 'rank_group_count': 3,
 'architecture_decision_id': 'src-objective-loss-contract-v1',
 'extension_requires_reason': True,
 'blocked_without_evidence': True,
 'decision_hash': 'f0d8963f64a4028f390b353db4e71d4bfb3c13afd8a22c3f414dada870d223a3',
 'registry_hash': '32ad6bf04474feac03d4b2e700cee7d0f777c8d41205a7f691b79960306c7b71',
 'artifacts': {'objective_losses': 'objective_losses.json',
  'objective_losses_csv': 'objective_losses.csv',
  'objective_loss_frame': 'objective_loss_frame.csv',
  'architecture_decision': 's

## 中文结论

- 做了什么：在 `src/objective` 中验证稳定 loss contract，把 MSE、classification、IC 和 ranking/Rank IC 指标统一投影成可最小化 loss，并把扩展原因写入 architecture extension registry。
- 核心指标：`objective.loss.v1` 输出 `mse_loss`、`classification_loss`、`ic_loss`、`ranking_loss`、`rank_group_count` 和原始 `objective.evaluation.v1`；本例 `ranking_loss` 约为 0，`reward_surface=retired`。
- regime 差异：本 notebook 只验证 objective loss 契约，不做 regime/window 切片；后续由 `src.analysis` 在不同 regime/window 中比较这些 loss 与 backtest 结果。
- 是否晋升：否。本卡晋升的是 `src/objective` 框架能力，不晋升任何模型或交易策略，也不进入 live broker。
- 风险：`ic_loss=1-IC`、`ranking_loss=1-RankIC` 是统一 loss contract，真实训练 loss 或 LambdaRankIC 近似需要额外研究和 license review；不要把它误当 RL reward。
- 下一步：让 QuantBench model matrix 和 dashboard export 读取 `objective.loss.v1` sidecar，与现有 objective score、promotion score 一起展示。